<a href="https://colab.research.google.com/github/yaswanthcs24/BIS-LAB/blob/main/1WN24CS133_CuckooSearchAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import numpy as np

# Knapsack data
weights = np.array([3, 2, 5, 4, 1, 3, 2])
values = np.array([40, 30, 50, 35, 20, 25, 45])
capacity = 15

# Fixed algorithm settings
MAX_ITER = 100
ALPHA = 0.1

# Fitness function
def fitness(solution):
    total_weight = np.sum(solution * weights)
    total_value = np.sum(solution * values)

    if total_weight > capacity:
        return 1000000

    return -total_value

# Convert continuous solution into binary solution
def decode(position):
    solution = (position > 0.5).astype(int)

    while np.sum(solution * weights) > capacity:
        selected = np.where(solution == 1)[0]
        ratios = values[selected] / weights[selected]
        remove = selected[np.argmin(ratios)]
        solution[remove] = 0

    return solution

# Levy Flight
def levy_flight(size):
    beta = 1.5
    u = np.random.normal(0, 1, size)
    v = np.random.normal(0, 1, size)

    return u / (np.abs(v) ** (1 / beta))

# Cuckoo Search Algorithm
def cuckoo_search(n, Pa):

    # Step 1: Initialize host nests
    nests = np.random.rand(n, len(weights))
    solutions = np.array([decode(x) for x in nests])
    scores = np.array([fitness(x) for x in solutions])

    best_index = np.argmin(scores)
    best_solution = solutions[best_index].copy()
    best_score = scores[best_index]

    # Step 2: Generate new solutions
    for iteration in range(MAX_ITER):

        for i in range(n):

            new_nest = nests[i] + ALPHA * levy_flight(len(weights))
            new_nest = np.clip(new_nest, 0, 1)

            new_solution = decode(new_nest)
            new_score = fitness(new_solution)

            # Step 3: Select random nest and compare
            j = np.random.randint(n)

            if new_score < scores[j]:
                nests[j] = new_nest
                solutions[j] = new_solution
                scores[j] = new_score

        # Step 4: Abandon worst nests
        count = max(1, int(Pa * n))
        worst = np.argsort(scores)[-count:]

        for j in worst:
            nests[j] = np.random.rand(len(weights))
            solutions[j] = decode(nests[j])
            scores[j] = fitness(solutions[j])

        # Step 5: Update best solution
        best_index = np.argmin(scores)

        if scores[best_index] < best_score:
            best_score = scores[best_index]
            best_solution = solutions[best_index].copy()

    # Step 6: Display result
    print("Selected Items:")

    for i in range(len(weights)):
        if best_solution[i] == 1:
            print("Item", i + 1,
                  "Weight:", weights[i],
                  "Value:", values[i])

    print("Total Weight:", np.sum(best_solution * weights))
    print("Maximum Value:", -best_score)


# Only TWO input parameters
cuckoo_search(10, 0.25)


Selected Items:
Item 1 Weight: 3 Value: 40
Item 2 Weight: 2 Value: 30
Item 4 Weight: 4 Value: 35
Item 5 Weight: 1 Value: 20
Item 6 Weight: 3 Value: 25
Item 7 Weight: 2 Value: 45
Total Weight: 15
Maximum Value: 195
